# Reproducing Maia-2 (NeurIPS 2024): predicting human chess moves

**Paper:** Z. Tang, D. Jiao, R. McIlroy-Young, J. Kleinberg, S. Sen and A. Anderson, *Maia-2: A Unified Model for Human-AI Alignment in Chess*, NeurIPS 2024. https://openreview.net/forum?id=XWlkhRn14K · arXiv: https://doi.org/10.48550/arXiv.2409.20553
**Official code:** https://github.com/CSSLab/maia2 (MIT), pinned to commit `67bee6c`
**Data:** Lichess rated Rapid games (https://database.lichess.org/), released by the authors through the official package (`maia2.dataset`).

**Task (prediction / classification):** given a chess position plus the ratings (Elo) of the player to move and of the opponent, predict the move that player actually made. Every possible move is a class, and the paper's main metric is **top-1 move-prediction accuracy**.

**Model:** a **ResNet CNN** encodes the 8×8 board (18 input channels). Skill-aware attention blocks then mix in embeddings of both players' ratings, and three heads output the move (policy), auxiliary move information and the game result (value).

**What this notebook does**
| Part | Experiment | Paper reference |
|---|---|---|
| A | Data audit: download, filter and split checks | Sec. 4, App. |
| B | **Reproduction:** official pretrained Maia-2 (Rapid) on the full Maia Testset (106,740 positions) | **Table 1** |
| C | **Own training:** train Maia-2 from scratch with the official training code on the official example training games | Sec. 3 method, small scale |
| D | Own analysis: the same pretrained model with the rating input removed (skill-blind) | related to Table 3 |
| E | Comparison table, difference from the paper, figures, and an evidence zip | — |

### How to run on Colab
1. **Runtime → Change runtime type → T4 GPU.**
2. First set `SMOKE_TEST = True` and do **Runtime → Run all** (about 10 minutes). This checks that everything works.
3. Then set `SMOKE_TEST = False` and **Run all** again for the real experiment. With `SAVE_TO_DRIVE = True`, all outputs go to Google Drive (`MyDrive/maia2-reproduction/`).
No accounts or tokens are needed: the data and model are public.

## 0. Configuration

In [ ]:
SMOKE_TEST = True        # True: tiny run to test the pipeline. False: the real experiment
SAVE_TO_DRIVE = True     # Colab: write all outputs to Google Drive so they survive a disconnect

# Official values come from maia2/configs/maia2-training-rapid.yaml. Values that differ are deviations and are listed in the report.
EPOCHS = 3               # official: 3
BATCH_SIZE = 2048        # official: 8192 on 2x A100 80 GB -> 2048 fits a 16 GB T4 (deviation)
CHUNK_SIZE = 1000        # games per preprocessing chunk (official 20000). Only changes how often the loss is logged
BALANCE_CAP = None       # official max_games_per_elo_range = 20 per chunk. None = keep all games (deviation, see Part C)
EVAL_BATCH = 1024
NUM_WORKERS = 2          # Colab has 2 CPU cores
SEED = 42
MAIA2_COMMIT = "67bee6ce7e2f264344fe6d2026a2e18c76a82efa"

if SMOKE_TEST:
    EPOCHS, CHUNK_SIZE = 1, 100
    SMOKE_TEST_POSITIONS, SMOKE_TRAIN_GAMES = 2000, 300

## 1. Setup and software environment (part e)

In [ ]:
import os, sys, json, time, platform, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/maia2-reproduction")
else:
    OUT_ROOT = Path("maia2-reproduction-outputs")
OUT = OUT_ROOT / ("smoke" if SMOKE_TEST else "full")
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
WORK = Path("work").resolve()
WORK.mkdir(exist_ok=True)

if IN_COLAB:
    # The official package pins torch 2.8 / Python <3.13. Install it without its dependencies so Colab's own
    # CUDA build of PyTorch stays in place; the versions actually used are recorded below.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--ignore-requires-python",
                    f"git+https://github.com/CSSLab/maia2.git@{MAIA2_COMMIT}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chess>=1.10.0,<2", "einops>=0.8.0,<1",
                    "gdown>=5.2.2,<6", "pyzstd>=0.19.1,<0.20"], check=True)
print("Outputs ->", OUT.resolve())

In [ ]:
import importlib.metadata as md_
import torch, numpy as np, pandas as pd, chess
env = dict(
    python=sys.version.split()[0], platform=platform.platform(),
    torch=torch.__version__, cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    gpu_memory_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else None,
    cpu_count=os.cpu_count(), numpy=np.__version__, pandas=pd.__version__,
    python_chess=md_.version("chess"), maia2=md_.version("maia2"), maia2_commit=MAIA2_COMMIT,
    colab=IN_COLAB, smoke_test=SMOKE_TEST,
    settings=dict(epochs=EPOCHS, batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE, balance_cap=BALANCE_CAP, seed=SEED),
)
(OUT / "environment.json").write_text(json.dumps(env, indent=2))
(OUT / "pip-freeze.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(json.dumps(env, indent=2))
if torch.cuda.is_available():
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Part A: dataset, preprocessing and splits (part d)

Both files come from the official package (`maia2.dataset`), which checks their SHA-256 checksums:
- **Test:** `example_test_dataset.csv`, one row per position (FEN board, move played, both ratings). The paper's filter keeps positions after the first 10 plies (`move_ply > 10`), which leaves the **Maia Testset** of Table 1 (paper: 106,740 positions).
- **Train:** `example_train_dataset.csv`, which is really a Lichess PGN file of real games. The official training code applies its own filters: rated Rapid only, no bot accounts, clock times present, skip the first 10 plies, and skip positions where the player has ≤ 30 s left.

**Skill groups (Table 1):** Skilled < 1600, Advanced 1600–1999, Master ≥ 2000, using the game's rounded rating (`rounded_elo`). This definition reproduces the paper's group sizes exactly; the next cell checks it.

In [ ]:
from maia2 import dataset
from maia2.main import game_filter
from maia2.utils import extract_clock_time
import chess.pgn

DATA = WORK / "maia2_data"
official_test = dataset.load_example_test_dataset(save_root=str(DATA))   # official loader: download + checksum + ply filter
raw_test = pd.read_csv(DATA / "example_test_dataset.csv")
test = raw_test[raw_test.move_ply > 10].reset_index(drop=True)            # identical filter, keeping the metadata columns
assert len(test) == len(official_test)

GROUPS = ["Skilled", "Advanced", "Master"]
test["group"] = pd.cut(test.rounded_elo, [0, 1600, 2000, 10_000], right=False, labels=GROUPS)
PAPER_COUNTS = {"Skilled": 56_812, "Advanced": 41_747, "Master": 8_181}
counts = test.group.value_counts().reindex(GROUPS).to_dict()
print("Raw test rows:", len(raw_test), "| games:", raw_test.game_id.nunique(), "| after move_ply > 10:", len(test))
print("Group sizes:", counts, "| paper:", PAPER_COUNTS)
assert counts == PAPER_COUNTS, "test set does not match the paper's Maia Testset"
print("Time controls:", test.type.value_counts().to_dict())

if SMOKE_TEST:
    test = pd.concat([g.sample(min(len(g), SMOKE_TEST_POSITIONS // 3), random_state=SEED)
                      for _, g in test.groupby("group", observed=True)]).reset_index(drop=True)
    print("SMOKE TEST: using", len(test), "test positions")

In [ ]:
train_pgn = Path(dataset.load_example_train_dataset(save_root=str(DATA)))

def train_data_audit(path, limit=None):
    """Apply the official game filter and position filters to count what the training code will use."""
    stats = dict(games=0, kept_games=0, positions_after_filters=0, dates=set(), game_ids=set())
    with open(path, encoding="utf-8") as fh:
        while (game := chess.pgn.read_game(fh)) is not None:
            stats["games"] += 1
            stats["dates"].add(game.headers.get("Date"))
            stats["game_ids"].add(game.headers.get("Site", "").rsplit("/", 1)[-1])
            if game_filter(game, "rapid"):
                stats["kept_games"] += 1
                for i, node in enumerate(game.mainline()):   # same rules as maia2.main.process_per_game
                    if i >= 10 and extract_clock_time(node.comment) > 30:
                        stats["positions_after_filters"] += 1
                    if i == 300:
                        break
            if limit and stats["games"] >= limit:
                break
    return stats

t0 = time.time()
tr = train_data_audit(train_pgn, SMOKE_TRAIN_GAMES if SMOKE_TEST else None)
overlap = len(tr["game_ids"] & set(raw_test.game_id))
print(f"Train PGN: {tr['games']} games, {tr['kept_games']} pass the official filter, "
      f"~{tr['positions_after_filters']:,} training positions; dates {sorted(tr['dates'])}; audit {time.time()-t0:.0f}s")
print("Games shared between train and test:", overlap)
assert overlap == 0, "leakage: a game appears in both train and test"

data_summary = dict(
    test=dict(raw_rows=len(raw_test), games=int(raw_test.game_id.nunique()), positions_after_filter=int(len(official_test)),
              group_counts=counts, paper_group_counts=PAPER_COUNTS, used_in_this_run=len(test)),
    train=dict(games=tr["games"], games_passing_official_filter=tr["kept_games"],
               positions_after_filters=tr["positions_after_filters"], dates=sorted(tr["dates"])),
    train_test_game_overlap=overlap,
)
(OUT / "data_summary.json").write_text(json.dumps(data_summary, indent=2))

In [ ]:
import matplotlib.pyplot as plt
# Validated categorical slots (1 blue, 2 orange, 3 aqua) + recessive ink
C1, C2, C3, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "figure.dpi": 110})

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
vc = test.rounded_elo.value_counts().sort_index()
ax.bar(vc.index, vc.values, width=80, color=C1)
for x in (1600, 2000):
    ax.axvline(x - 50, color=MUTED, ls="--", lw=1)
ax.set_xlabel("Rounded rating (Elo)"); ax.set_ylabel("Test positions")
ax.set_title("Test set by rating (dashed lines = Skilled | Advanced | Master)", loc="left")
plt.tight_layout(); plt.savefig(FIG / "test_rating_distribution.png", dpi=200); plt.show()

## 3. Part B: reproduce Table 1 with the official pretrained model (parts a, e, f)

`model.from_pretrained("rapid")` downloads the authors' released Rapid checkpoint (checksum-verified), and `inference.inference_batch` is the official evaluation routine: masked softmax over legal moves, then top-1. Accuracy is reported for each skill group plus their average, which is how the paper's "Avg" column is computed (51.72, 54.15 and 53.87 average to 53.25). 95% Wilson confidence intervals are added.

In [ ]:
from maia2 import model as maia_model, inference
from statsmodels.stats.proportion import proportion_confint

PAPER_T1 = {  # Tang et al. (NeurIPS 2024), Table 1, top-1 accuracy (%) on the Maia Testset
    "Maia-2": dict(Skilled=51.72, Advanced=54.15, Master=53.87, Avg=53.25),
    "Maia 1500 (best Maia-1)": dict(Skilled=50.79, Advanced=52.61, Master=50.76, Avg=51.39),
}

def predict(model_, df, active_elo=None, opponent_elo=None):
    """Official batch inference; optional fixed ratings for the skill-blind analysis."""
    inp = df[["board", "move", "active_elo", "opponent_elo"]].copy()
    if active_elo is not None:
        inp["active_elo"], inp["opponent_elo"] = active_elo, opponent_elo
    t0 = time.time()
    out, official_acc = inference.inference_batch(inp, model_, verbose=1, batch_size=EVAL_BATCH, num_workers=NUM_WORKERS)
    ranked = out.move_probs.map(list)
    res = pd.DataFrame({
        "group": df.group.values, "rounded_elo": df.rounded_elo.values, "move": df.move.values,
        "pred": ranked.map(lambda m: m[0]).values,
        "p_pred": out.move_probs.map(lambda d: next(iter(d.values()))).values,
        "p_true": [d.get(m, 0.0) for d, m in zip(out.move_probs, df.move)],
        "top3": [m in r[:3] for m, r in zip(df.move, ranked)],
    })
    res["correct"] = res.pred == res.move
    print(f"official inference_batch accuracy {official_acc:.4f} | {len(df)} positions in {time.time()-t0:.0f}s")
    return res

def summarize(res, name):
    row = {"Model": name}
    for g in GROUPS:
        r = res[res.group == g]
        lo, hi = proportion_confint(r.correct.sum(), len(r), method="wilson")
        row[g] = 100 * r.correct.mean()
        row[f"{g} CI"] = f"{100*lo:.2f}–{100*hi:.2f}"
        row[f"{g} n"] = len(r)
    row["Avg"] = np.mean([row[g] for g in GROUPS])
    row["Overall (all positions)"] = 100 * res.correct.mean()
    row["Top-3 avg"] = np.mean([100 * res[res.group == g].top3.mean() for g in GROUPS])
    return row

pretrained = maia_model.from_pretrained(type="rapid", device="auto", save_root=str(WORK / "models"))
pre_res = predict(pretrained, test)
pre_res.to_csv(OUT / "predictions_pretrained.csv", index=False)
pre_row = summarize(pre_res, "Maia-2 official checkpoint (reproduced)")

t1 = pd.DataFrame([{"Model": f"{k} (paper)", **v} for k, v in PAPER_T1.items()] + [pre_row])
t1["Δ Avg vs paper (pp)"] = t1.Avg - PAPER_T1["Maia-2"]["Avg"]
t1.to_csv(OUT / "table1_reproduction.csv", index=False)
display(t1[["Model", *GROUPS, "Avg", "Δ Avg vs paper (pp)", "Overall (all positions)", "Top-3 avg"]].round(2))
display(pd.DataFrame([{g: pre_row[f"{g} CI"] for g in GROUPS}], index=["95% CI (reproduced)"]))

## 4. Part C: train Maia-2 ourselves with the official training code (parts e, f)

The official `maia2.train.run` is used without changes. It reads monthly Lichess archives named `lichess_db_standard_rated_YYYY-MM.pgn.zst`, so the official example training PGN (all games from 2018-04-30) is compressed into that format as month `2018-04`. The model is the same architecture as the release (`maia2-training-rapid.yaml`), trained **from random initialisation**.

**Deviations from the paper (hardware/data limits):**
- **Data:** the paper used 169M games (9.1B positions) and 13 days on 2× A100. We use the ~28k released example games, roughly 0.02% of the paper's data.
- **Batch size:** 2048 instead of 8192, to fit a 16 GB T4.
- **Balancing cap:** the official per-chunk cap of 20 games per rating pair is meant for monthly files of millions of games. On 28k games it would throw most of them away, so `BALANCE_CAP = None` keeps all of them. Setting it to 20 runs the official value.

Checkpoints are saved after each epoch, and each one is evaluated on the same test set. There is no checkpoint selection: the final epoch is reported, and the curve is for monitoring only.

In [ ]:
import pyzstd, re

DATA_ROOT = WORK / "lichess_data"; DATA_ROOT.mkdir(exist_ok=True)
RUN_DIR = WORK / f"own_training_{time.strftime('%Y%m%d-%H%M%S')}"
month_pgn = DATA_ROOT / "lichess_db_standard_rated_2018-04.pgn"
for p in DATA_ROOT.glob("lichess_db_standard_rated_2018-04.pgn*"):
    p.unlink()                                   # fresh copy each run

text = train_pgn.read_text(encoding="utf-8")
if SMOKE_TEST:                                   # keep only the first N games
    starts = [m.start() for m in re.finditer(r"^\[Event ", text, flags=re.M)]
    text = text[: starts[SMOKE_TRAIN_GAMES]] if len(starts) > SMOKE_TRAIN_GAMES else text
Path(str(month_pgn) + ".zst").write_bytes(pyzstd.compress(text.encode("utf-8")))

overrides = dict(
    data_root=str(DATA_ROOT), save_root=str(RUN_DIR), game_type="rapid", seed=SEED,
    start_year=2018, start_month=4, end_year=2018, end_month=4,
    max_epochs=EPOCHS, batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE, num_workers=NUM_WORKERS, verbose=0,
    max_games_per_elo_range=BALANCE_CAP if BALANCE_CAP is not None else 10**9,
)
Path("train_maia2.py").write_text("""
import json, sys
from importlib.resources import as_file, files
from maia2 import train, utils

def main():
    with as_file(files("maia2.configs").joinpath("maia2-training-rapid.yaml")) as path:
        cfg = utils.parse_args(path)          # official Rapid preset
    for key, value in json.loads(sys.argv[1]).items():
        setattr(cfg, key, value)              # overrides listed in the notebook
    train.run(cfg, device="auto")

if __name__ == "__main__":                    # required: the official trainer spawns worker processes
    main()
""")
(OUT / "training_overrides.json").write_text(json.dumps(overrides, indent=2))
print(json.dumps(overrides, indent=2))

In [ ]:
log_path = OUT / "train.log"
t0 = time.time()
with open(log_path, "w") as log:
    proc = subprocess.Popen([sys.executable, "train_maia2.py", json.dumps(overrides)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        line = f"{time.strftime('%H:%M:%S')} {line}"
        log.write(line); log.flush()
        if not line.split(" ", 1)[1].startswith(("\t", "  ", "(", ")")):   # keep the cell output readable; the full log is saved
            print(line, end="")
    rc = proc.wait()
train_minutes = (time.time() - t0) / 60
print(f"exit code {rc} | training wall time {train_minutes:.1f} min")
assert rc == 0, f"training failed, see {log_path}"

In [ ]:
# Training curves from the official trainer's log (one point per preprocessed chunk batch)
pat = re.compile(r"\[# Loss\]: ([\d.]+) \| \[# Loss MAIA\]: ([\d.]+) \| \[# Loss Side Info\]: ([\d.]+) \| \[# Loss Value\]: ([\d.]+)")
rows, epoch = [], 0
for line in log_path.read_text().splitlines():
    if re.search(r" Epoch (\d+)$", line):
        epoch = int(line.rsplit(" ", 1)[1])
    elif (m := pat.search(line)):
        rows.append(dict(epoch=epoch, total=float(m[1]), move=float(m[2]), side_info=float(m[3]), value=float(m[4])))
curve = pd.DataFrame(rows); curve["step"] = range(1, len(curve) + 1)
curve.to_csv(OUT / "training_curve.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, cols, title in [(axes[0], ["move"], "Move-prediction loss (cross-entropy)"),
                        (axes[1], ["total", "side_info", "value"], "Total and auxiliary losses")]:
    ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
    for col, color in zip(cols, [C1, C2, C3]):
        ax.plot(curve.step, curve[col], color=color, lw=2, label=col.replace("_", " "))
    for e in curve.groupby("epoch").step.min().iloc[1:]:
        ax.axvline(e - 0.5, color=MUTED, ls=":", lw=1)
    ax.set_xlabel("Chunk batch (dotted line = new epoch)"); ax.set_title(title, loc="left")
    if len(cols) > 1:
        ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG / "training_loss.png", dpi=200); plt.show()

In [ ]:
from maia2 import utils as mutils
from maia2.main import MAIA2Model
from maia2.train import load_model_state_dict
from importlib.resources import as_file, files

with as_file(files("maia2.configs").joinpath("maia2-training-rapid.yaml")) as path:
    train_cfg = mutils.parse_args(path)
for k, v in overrides.items():
    setattr(train_cfg, k, v)
all_moves, elo_dict = mutils.get_all_possible_moves(), mutils.create_elo_dict()
device = "cuda" if torch.cuda.is_available() else "cpu"

epoch_rows, own_res = [], None
ckpts = sorted(RUN_DIR.rglob("epoch_*_2018-04.pgn.pt"), key=lambda p: int(p.name.split("_")[1]))
assert ckpts, f"no checkpoints found in {RUN_DIR}"
for ckpt_path in ckpts:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    net = MAIA2Model(len(all_moves), elo_dict, train_cfg)
    load_model_state_dict(net, ckpt["model_state_dict"])
    net = net.to(device).eval()
    ep = int(ckpt_path.name.split("_")[1])
    print(f"Epoch {ep}: trained on {ckpt['accumulated_samples']:,} positions / {ckpt['accumulated_games']:,} games")
    own_res = predict(net, test)
    epoch_rows.append({"epoch": ep, "positions_seen": ckpt["accumulated_samples"], **summarize(own_res, f"Own training, epoch {ep}")})
own_res.to_csv(OUT / "predictions_own_trained.csv", index=False)
epochs_df = pd.DataFrame(epoch_rows)
epochs_df.to_csv(OUT / "own_training_by_epoch.csv", index=False)
own_row = epoch_rows[-1] | {"Model": f"Own training from scratch (final epoch {epoch_rows[-1]['epoch']})"}
display(epochs_df[["epoch", "positions_seen", *GROUPS, "Avg", "Top-3 avg"]].round(2))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
ax.plot(epochs_df.epoch, epochs_df.Avg, color=C3, lw=2, marker="o", ms=8, label="Own training (avg of groups)")
ax.axhline(PAPER_T1["Maia-2"]["Avg"], color=C1, lw=2, label="Paper Maia-2 (53.25)")
for x, y in zip(epochs_df.epoch, epochs_df.Avg):
    ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", color=INK)
ax.set_xticks(epochs_df.epoch); ax.set_xlabel("Epoch"); ax.set_ylabel("Top-1 accuracy (%)"); ax.set_ylim(0, 60)
ax.set_title("Own training: test accuracy after each epoch (monitoring only)", loc="left")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(FIG / "own_training_accuracy_by_epoch.png", dpi=200); plt.show()

## 5. Part D: skill-blind analysis (own analysis)

This analysis is our own. The paper's central claim is that conditioning on both players' ratings (skill-aware attention) makes the model predict human moves better. As a simple test that needs no retraining, the **same pretrained model** is evaluated with every player's rating set to 1500. If the rating input matters, accuracy should fall, mostly for players far from 1500. This is **not** the paper's Table 3 ablation, which retrains without the attention mechanism.

In [ ]:
blind_res = predict(pretrained, test, active_elo=1500, opponent_elo=1500)
blind_res.to_csv(OUT / "predictions_skill_blind.csv", index=False)
blind_row = summarize(blind_res, "Maia-2 checkpoint, ratings fixed to 1500 (skill-blind)")
display(pd.DataFrame([pre_row, blind_row])[["Model", *GROUPS, "Avg"]].round(2))

## 6. Part E: comparison with the paper (parts f, g)

**Main metric:** top-1 accuracy, averaged over the three skill groups as in Table 1. **Difference = reproduced − paper** (percentage points).

In [ ]:
paper = PAPER_T1["Maia-2"]
final = pd.DataFrame([{"Model": "Maia-2 (paper, Table 1)", **paper}, pre_row, own_row, blind_row])
for g in GROUPS + ["Avg"]:
    final[f"Δ {g} (pp)"] = final[g] - paper[g]
final.to_csv(OUT / "final_comparison.csv", index=False)
display(final[["Model", *GROUPS, "Avg", *[f"Δ {g} (pp)" for g in GROUPS + ["Avg"]]]].round(2))
summary = dict(main_metric="top-1 accuracy, mean of skill groups", paper_avg=paper["Avg"],
               reproduced_avg=round(pre_row["Avg"], 2), difference_pp=round(pre_row["Avg"] - paper["Avg"], 2),
               own_training_avg=round(own_row["Avg"], 2), skill_blind_avg=round(blind_row["Avg"], 2),
               training_minutes=round(train_minutes, 1), smoke_test=SMOKE_TEST)
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

In [ ]:
cats = GROUPS + ["Avg"]
series = [("Paper (Table 1)", [paper[c] for c in cats], C1),
          ("Reproduced (official checkpoint)", [pre_row[c] for c in cats], C2),
          ("Own training (small data)", [own_row[c] for c in cats], C3)]
x = np.arange(len(cats)); w = 0.26
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
for i, (name, vals, color) in enumerate(series):
    xs = x + (i - 1) * w
    ax.bar(xs, vals, w - 0.03, color=color, label=name)
    for xi, v in zip(xs, vals):
        ax.text(xi, v + 0.8, f"{v:.1f}", ha="center", va="bottom", fontsize=8.5, color=INK)
ax.set_xticks(x, ["Skilled\n<1600", "Advanced\n1600–1999", "Master\n≥2000", "Average"])
ax.set_ylabel("Top-1 move accuracy (%)"); ax.set_ylim(0, 65)
ax.set_title("Maia-2 move-prediction accuracy: paper vs reproduction", loc="left")
ax.legend(frameon=False, ncol=3, loc="upper left", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "paper_vs_reproduced.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
d = [pre_row[c] - paper[c] for c in cats]
ax.bar(x, d, 0.5, color=C2)
ax.axhline(0, color=INK, lw=1)
for xi, v in zip(x, d):
    ax.text(xi, v + (0.05 if v >= 0 else -0.05), f"{v:+.2f} pp", ha="center", va="bottom" if v >= 0 else "top", color=INK)
lim = max(1.0, max(abs(v) for v in d) * 1.6)
ax.set_ylim(-lim, lim); ax.set_xticks(x, cats); ax.set_ylabel("Δ accuracy (pp)")
ax.set_title("Difference: reproduced − paper (main metric)", loc="left")
plt.tight_layout(); plt.savefig(FIG / "difference_vs_paper.png", dpi=200); plt.show()

In [ ]:
def by_rating(res):
    return res.groupby("rounded_elo").correct.agg(["mean", "size"]).query("size >= 50")["mean"] * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
for res, name, color in [(pre_res, "Maia-2 official checkpoint", C2), (own_res, "Own training", C3),
                         (blind_res, "Checkpoint, skill-blind (1500)", C1)]:
    s = by_rating(res)
    ax.plot(s.index, s.values, color=color, lw=2, marker="o", ms=5, label=name)
for xv in (1600, 2000):
    ax.axvline(xv - 50, color=MUTED, ls="--", lw=1)
ax.set_xlabel("Rounded rating (Elo)"); ax.set_ylabel("Top-1 accuracy (%)")
ax.set_title("Accuracy by player rating (bins with ≥ 50 positions)", loc="left")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "accuracy_by_rating.png", dpi=200); plt.show()

## 7. Execution evidence

The zip contains `environment.json`, `pip-freeze.txt`, `data_summary.json`, `training_overrides.json`, the timestamped `train.log`, every `predictions_*.csv`, the result tables (`table1_reproduction.csv`, `own_training_by_epoch.csv`, `final_comparison.csv`, `summary.json`) and `figures/`. Download it (or take it from Drive) and add it to the group's GitHub repository with this notebook. Take screenshots of the cell outputs as they run.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_ROOT / f"maia2-results-{'smoke' if SMOKE_TEST else 'full'}"), "zip", OUT)
print("Saved", archive)
if IN_COLAB and not SAVE_TO_DRIVE:
    from google.colab import files as colab_files
    colab_files.download(archive)